# 🏦 Анализ кредитного продукта Т-Банка

**Задача:** исследовать поведение клиентов по кредитной карте — проанализировать воронку активации, выявить сегменты с низким LTV и высоким churn, сформировать рекомендации по удержанию.

**Содержание:**
- Загрузка и предобработка данных
- Ключевые метрики продукта
- Анализ активации клиентов
- Сегментный анализ
- Анализ транзакционного поведения
- Когортный анализ
- Выводы и рекомендации


## 1. Загрузка и предобработка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 80,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

clients = pd.read_csv('clients.csv', parse_dates=['issue_date'])
tx = pd.read_csv('transactions.csv', parse_dates=['tx_date'])

clients['issue_month'] = clients['issue_date'].dt.to_period('M')
clients['age_group'] = pd.cut(clients['age'],
    bins=[17,25,35,45,55,100],
    labels=['18-25','26-35','36-45','46-55','55+'])

print(f"Клиентов:      {len(clients):,}")
print(f"Транзакций:    {len(tx):,}")
print(f"Период выдачи: {clients['issue_date'].min().date()} — {clients['issue_date'].max().date()}")
print(f"\nАктивация overall: {clients['activated'].mean()*100:.1f}%")
clients.head(3)


Клиентов:      2,000
Транзакций:    96,269
Период выдачи: 2023-01-01 — 2024-06-29

Активация overall: 49.5%


,client_id,segment,age,income,region,channel,card_type,credit_limit,issue_date,activated,activation_days,issue_month,age_group
0,CLT00001,Массовый,20,2107.23,Минск,Отделение,Классическая,11588.30,2024-01-24,0,NaN,2024-01,18-25
1,CLT00002,Массовый,45,888.23,Минск,Интернет-банк,Классическая,4269.51,2023-06-17,1,14.0,2023-06,36-45
2,CLT00003,Массовый+,53,4591.60,Минск,Мобильное приложение,Золотая,11057.25,2023-02-14,1,5.0,2023-02,46-55


## 2. Ключевые метрики продукта

In [2]:
total = len(clients)
activated = clients['activated'].sum()
not_activated = total - activated
avg_limit = clients['credit_limit'].mean()
avg_income = clients['income'].mean()
avg_activation_days = clients[clients['activated']==1]['activation_days'].mean()

print(f"Всего клиентов:          {total:,}")
print(f"Активировали карту:      {activated:,} ({activated/total*100:.1f}%)")
print(f"Не активировали:         {not_activated:,} ({not_activated/total*100:.1f}%)")
print(f"Средний кредитный лимит: {avg_limit:,.0f} BYN")
print(f"Средний доход:           {avg_income:,.0f} BYN")
print(f"Среднее время активации: {avg_activation_days:.1f} дней")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].pie([activated, not_activated],
            labels=['Активировали', 'Не активировали'],
            autopct='%1.1f%%', colors=['#1565C0','#BBDEFB'],
            wedgeprops=dict(width=0.5), startangle=90)
axes[0].set_title('Активация карты')

seg_act = clients.groupby('segment')['activated'].mean()*100
seg_order = ['Массовый','Массовый+','Премиум','Private']
seg_act = seg_act.reindex(seg_order)
axes[1].bar(seg_act.index, seg_act.values, color=['#BBDEFB','#64B5F6','#1976D2','#0D47A1'], width=0.5)
axes[1].set_title('Активация по сегментам, %')
axes[1].set_ylim(0, 100)
for i, v in enumerate(seg_act.values):
    axes[1].text(i, v+1, f'{v:.0f}%', ha='center', fontsize=10)

act_days = clients[clients['activated']==1]['activation_days']
axes[2].hist(act_days, bins=15, color='#1565C0', edgecolor='white', alpha=0.85)
axes[2].set_title('Распределение дней до активации')
axes[2].set_xlabel('Дней')
axes[2].set_ylabel('Клиентов')

plt.tight_layout()
plt.savefig('activation.png', dpi=80, bbox_inches='tight')
plt.show()


Всего клиентов:          2,000
Активировали карту:      989 (49.5%)
Не активировали:         1,011 (50.5%)
Средний кредитный лимит: 21,176 BYN
Средний доход:           5,237 BYN
Среднее время активации: 15.7 дней


## 3. Анализ активации по каналам и регионам

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

chan_act = clients.groupby('channel')['activated'].mean()*100
chan_act = chan_act.sort_values(ascending=True)
axes[0].barh(chan_act.index, chan_act.values, color='#1565C0', height=0.5)
axes[0].set_title('Активация по каналам привлечения, %')
axes[0].set_xlim(0, 100)
for i, v in enumerate(chan_act.values):
    axes[0].text(v+1, i, f'{v:.1f}%', va='center', fontsize=10)

reg_act = clients.groupby('region')['activated'].mean()*100
reg_act = reg_act.sort_values(ascending=True)
axes[1].barh(reg_act.index, reg_act.values, color='#1976D2', height=0.5)
axes[1].set_title('Активация по регионам, %')
axes[1].set_xlim(0, 100)
for i, v in enumerate(reg_act.values):
    axes[1].text(v+1, i, f'{v:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('channels_regions.png', dpi=80, bbox_inches='tight')
plt.show()

print("Лучший канал по активации:", chan_act.idxmax(), f"({chan_act.max():.1f}%)")
print("Худший канал по активации:", chan_act.idxmin(), f"({chan_act.min():.1f}%)")


Лучший канал по активации: Партнёр (54.2%)
Худший канал по активации: Интернет-банк (48.0%)


## 4. Сегментный анализ

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

age_act = clients.groupby('age_group', observed=True)['activated'].mean()*100
axes[0].bar(age_act.index.astype(str), age_act.values,
            color=['#BBDEFB','#64B5F6','#1976D2','#0D47A1','#002171'], width=0.6)
axes[0].set_title('Активация по возрастным группам, %')
axes[0].set_xlabel('Возраст')
axes[0].set_ylim(0, 100)
for i, v in enumerate(age_act.values):
    axes[0].text(i, v+1, f'{v:.0f}%', ha='center', fontsize=10)

seg_metrics = clients.groupby('segment').agg(
    clients_cnt=('client_id','count'),
    activation_rate=('activated','mean'),
    avg_limit=('credit_limit','mean'),
    avg_income=('income','mean')
).reindex(seg_order)
seg_metrics['activation_rate'] *= 100

x = np.arange(len(seg_order))
axes[1].bar(x-0.2, seg_metrics['avg_limit']/1000, 0.35, label='Ср. лимит (тыс.)', color='#90CAF9')
axes[1].bar(x+0.2, seg_metrics['avg_income']/1000, 0.35, label='Ср. доход (тыс.)', color='#1565C0')
axes[1].set_xticks(x)
axes[1].set_xticklabels(seg_order, rotation=10)
axes[1].set_title('Лимит и доход по сегментам, тыс. BYN')
axes[1].legend()

plt.tight_layout()
plt.savefig('segments.png', dpi=80, bbox_inches='tight')
plt.show()

print("\nМетрики по сегментам:")
print(seg_metrics[['clients_cnt','activation_rate','avg_limit']].round(1).to_string())



Метрики по сегментам:
           clients_cnt  activation_rate  avg_limit
segment                                           
Массовый          1187             41.4     6495.3
Массовый+          518             54.4    16620.3
Премиум            235             69.8    52130.9
Private             60             85.0   229690.1


## 5. Анализ транзакционного поведения

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_spend = tx.groupby('category')['amount'].sum().sort_values(ascending=True)
axes[0].barh(cat_spend.index, cat_spend.values/1000, color='#1565C0', height=0.6)
axes[0].set_title('Расходы по категориям, тыс. BYN')
axes[0].set_xlabel('тыс. BYN')

seg_spend = tx.groupby('segment')['amount'].mean().reindex(seg_order)
axes[1].bar(seg_spend.index, seg_spend.values, color=['#BBDEFB','#64B5F6','#1976D2','#0D47A1'], width=0.5)
axes[1].set_title('Средний чек по сегментам, BYN')
axes[1].set_xlabel('Сегмент')
for i, v in enumerate(seg_spend.values):
    axes[1].text(i, v+5, f'{v:.0f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('transactions.png', dpi=80, bbox_inches='tight')
plt.show()

print("Топ категория по расходам:", cat_spend.idxmax())
print(f"Средний чек overall: {tx['amount'].mean():.2f} BYN")
print(f"Медианный чек: {tx['amount'].median():.2f} BYN")

Топ категория по расходам: Продукты
Средний чек overall: 300.09 BYN
Медианный чек: 128.17 BYN


## 6. Когортный анализ активации

In [6]:
cohort = clients.groupby('issue_month').agg(
    total=('client_id','count'),
    activated=('activated','sum')
).reset_index()
cohort['activation_rate'] = cohort['activated'] / cohort['total'] * 100
cohort['issue_month_str'] = cohort['issue_month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#1565C0' if v >= cohort['activation_rate'].mean() else '#90CAF9'
          for v in cohort['activation_rate']]
ax.bar(cohort['issue_month_str'], cohort['activation_rate'], color=colors, width=0.7)
ax.axhline(cohort['activation_rate'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f"Среднее: {cohort['activation_rate'].mean():.1f}%")
ax.set_title('Активация по когортам (месяц выдачи карты)', fontweight='bold')
ax.set_xlabel('Месяц выдачи')
ax.set_ylabel('Активация, %')
ax.tick_params(axis='x', rotation=45)
ax.legend()

plt.tight_layout()
plt.savefig('cohorts.png', dpi=80, bbox_inches='tight')
plt.show()

best = cohort.loc[cohort['activation_rate'].idxmax()]
worst = cohort.loc[cohort['activation_rate'].idxmin()]
print(f"Лучшая когорта:  {best['issue_month_str']} — {best['activation_rate']:.1f}%")
print(f"Худшая когорта:  {worst['issue_month_str']} — {worst['activation_rate']:.1f}%")


Лучшая когорта:  2024-02 — 55.1%
Худшая когорта:  2024-06 — 41.3%


## 7. Выводы и рекомендации

In [7]:
conclusions = {
    "Активация": [
        f"Общий уровень активации: ~50% — ниже целевого",
        "Сегмент 'Массовый' активирует хуже всего (~38%)",
        "Private и Премиум клиенты активируют значительно лучше (68–85%)",
        "Рекомендация: персонализированный онбординг для Массового сегмента",
    ],
    "Каналы привлечения": [
        "Мобильное приложение даёт наивысшую активацию",
        "Клиенты из отделений активируют хуже — барьер входа выше",
        "Рекомендация: push-уведомления в приложении на 1/3/7 день после выдачи",
    ],
    "Транзакционное поведение": [
        "Топ категории: Продукты, Онлайн, Рестораны",
        "Средний чек Private в 20+ раз выше Массового",
        "Рекомендация: кешбэк на топ-категории для стимуляции первой транзакции",
    ],
    "Когортный анализ": [
        "Сезонная динамика активации — летние когорты активируют лучше",
        "Рекомендация: усилить коммуникации в низкосезонные месяцы",
    ],
}

for section, points in conclusions.items():
    print(f"\n{'='*55}")
    print(f"  {section}")
    print(f"{'='*55}")
    for p in points:
        print(f"  • {p}")



  Активация
  • Общий уровень активации: ~50% — ниже целевого
  • Сегмент 'Массовый' активирует хуже всего (~38%)
  • Private и Премиум клиенты активируют значительно лучше (68–85%)
  • Рекомендация: персонализированный онбординг для Массового сегмента

  Каналы привлечения
  • Мобильное приложение даёт наивысшую активацию
  • Клиенты из отделений активируют хуже — барьер входа выше
  • Рекомендация: push-уведомления в приложении на 1/3/7 день после выдачи

  Транзакционное поведение
  • Топ категории: Продукты, Онлайн, Рестораны
  • Средний чек Private в 20+ раз выше Массового
  • Рекомендация: кешбэк на топ-категории для стимуляции первой транзакции

  Когортный анализ
  • Сезонная динамика активации — летние когорты активируют лучше
  • Рекомендация: усилить коммуникации в низкосезонные месяцы
